# W14 正式推論示範
主要 final：Kaggle 58424527。使用 Conda kernel，設定 SOLAR_IMAGES_DIR 與 SOLAR_OUTPUT_DIR，依序執行全部 cell。預設 SOLAR_FINAL=w14，輸出 w14.csv；切換 W6 時使用全新目錄。Notebook 會驗證全部權重分片、重建模型並執行八階段 launcher。分環境 interpreter 可由 SOLAR_DENSE_PYTHON、SOLAR_SUPPORT_PYTHON、SOLAR_SCORE_PYTHON、SOLAR_REFINER_PYTHON 設定。每階段 log／完成收據保存在 cache；此 notebook 不提交 Kaggle。


In [ ]:
from pathlib import Path
import os, sys, subprocess, csv, time, platform
root = Path.cwd().resolve()
if root.name == "notebooks": root = root.parent
sys.path.insert(0, str(root))
from merge_weights import merge
images = Path(os.environ["SOLAR_IMAGES_DIR"]).resolve()
output = Path(os.environ.get("SOLAR_OUTPUT_DIR", "run_demo")).resolve()
output.mkdir(parents=True, exist_ok=True)
models = output / "models"
started = time.monotonic()
merge(root, models, os.environ.get("SOLAR_RELEASE_RAW_BASE_URL"))
assert images.is_dir()
print("SHARDS_AND_MODELS_SHA_VERIFIED", platform.machine())


In [ ]:
final = os.environ.get("SOLAR_FINAL", "w14")
assert final in ("w6", "w14")
filename = final + ".csv"
command = [sys.executable, str(root / "scripts/demo_w6.py"),
           "--images", str(images), "--models", str(models),
           "--cache", str(output / "cache"), "--output", str(output / filename),
           "--final", final, "--dense-long2x-tta", "--postcal-w2", "--device", "cuda"]
subprocess.run(command, cwd=root, check=True)


In [ ]:
for filename in (filename,):
    with (output / filename).open(newline="") as stream:
        reader = csv.DictReader(stream)
        assert reader.fieldnames == ["filament_id", "segmentation_rle"]
        rows = list(reader)
    assert len({row["filament_id"] for row in rows}) == len(rows)
    print(filename, len(rows))
print("TOTAL_SECONDS", time.monotonic() - started)
